# LangGraph Voice Translation Agent (MLX Native + Local LLM)

This notebook demonstrates a professional voice translation pipeline using:
1. **mlx-audio 0.4.0**: Native ASR (Qwen3-ASR) and TTS (Qwen3-TTS) on Apple Silicon.
2. **LangChain + Local LLM**: Qwen-based translation via an OpenAI-compatible API.

## 1. Setup and Dependencies

Ensure your environment is set up with the required libraries:
```bash
uv add langgraph mlx-audio==0.2.9 huggingface_hub langchain-openai transformers torch librosa
```

### Local LLM Requirement
This agent assumes a local OpenAI-compatible API is running (e.g., via vLLM, Ollama, or LM Studio) at `http://localhost:8000/v1`.

In [1]:
import os
import sys
from typing import TypedDict, Annotated, Dict, Optional
from enum import Enum
import asyncio
from langgraph.graph import StateGraph, END

# Add src to path if needed
sys.path.append(os.path.abspath("."))
from pipeline import VoiceTranslationPipeline, Language


## 2. Define Agent State

In [2]:
def dict_reducer(a: Dict, b: Dict | None) -> Dict:
    if b is None:
        return a
    return {**a, **b}


class AgentState(TypedDict):
    audio_path: str
    src_lang: str
    tgt_lang: str
    out_audio_path: str
    original_text: str          # raw ASR transcription (source language)
    translated_text: str        # final target-language text
    ref_audio_path: Optional[str]
    # Merge dicts from multiple nodes instead of “only one value per step”
    metrics: Annotated[Dict[str, float], dict_reducer]


## 3. Implement Workflow Nodes

In [3]:
pipeline = VoiceTranslationPipeline()
# ---------------------------------------------------------------------------
# Nodes
# ---------------------------------------------------------------------------
async def stt_node(state: AgentState):
    import time
    print("--- STT NODE (MLX NATIVE) ---")
    start = time.time()

    # 1) Transcribe audio (blocking, heavy) offloaded to worker thread
    original = await asyncio.to_thread(
        pipeline.transcribe_audio,
        state["audio_path"],
        state["src_lang"],
    )
    end = time.time()

    return {
        "original_text": original,
        "metrics": {
            "stt_time": end - start,
        },
    }


async def mt_node(state: AgentState):
    import time
    print("--- MT NODE (ENGLISH -> TARGET, LOCAL LLM) ---")
    start = time.time()

    # Always translate from English to target language
    translated = pipeline.translate_text(
        state["original_text"],
        state["src_lang"],
        state["tgt_lang"],
    )

    return {
        "translated_text": translated,
        "metrics": {"mt_time": time.time() - start},
    }


async def voice_clone_node(state: AgentState):
    """
    Prepare a 3-second reference clip for voice cloning, if source > 3 seconds.
    Runs concurrently with MT.
    """
    import time

    print("--- VOICE CLONE NODE (PREPARE REF CLIP) ---")
    start = time.time()

    # Use the output directory directly for the ref clip
    work_dir = state["out_audio_path"] or "."
    ref_audio_path = await asyncio.to_thread(
        pipeline.prepare_voice_clone,
        state["audio_path"],
        work_dir,
    )

    return {
        "ref_audio_path": ref_audio_path,
        "metrics": {"voice_clone_prep_time": time.time() - start},
    }


async def tts_node(state: AgentState):
    import time
    print("--- TTS NODE (MLX NATIVE, OPTIONAL VOICE CLONE) ---")
    start = time.time()

    # pass english only ref text if source lang is english then original else translated
    ref_en = ""
    if state["src_lang"] == "en":
        ref_en = state["original_text"]
    else:
        ref_en = state["translated_text"]
    ref_text = ref_en.split(".")[0] if "." in ref_en else ref_en

    await pipeline.text_to_speech(
        text=state["translated_text"],
        tgt_lang=state["tgt_lang"],
        output_path=state["out_audio_path"],
        ref_audio=state.get("ref_audio_path"),
        ref_text=ref_text,
    )

    return {
        "metrics": {"tts_time": time.time() - start},
    }


async def cleanup_node(state: AgentState):
    print("--- CLEANUP NODE ---")

    # Delete temporary voice clone clip if it exists
    ref_path = state.get("ref_audio_path")
    if ref_path and os.path.exists(ref_path):
        try:
            os.remove(ref_path)
            print(f"Deleted temporary voice reference clip: {ref_path}")
        except OSError as e:
            print(f"Failed to delete ref clip {ref_path}: {e}")

    pipeline.clear_memory()
    return state


Initializing Native MLX-Audio Pipeline
Models directory: /Users/nitinkumar/Work/letsTalkVoiceAgent/models
Using local model: /Users/nitinkumar/Work/letsTalkVoiceAgent/models/whisper-tiny
Using local model: /Users/nitinkumar/Work/letsTalkVoiceAgent/models/whisper-tiny-hi
Loading English Whisper from: /Users/nitinkumar/Work/letsTalkVoiceAgent/models/whisper-tiny
Loading Hindi Whisper from: /Users/nitinkumar/Work/letsTalkVoiceAgent/models/whisper-tiny-hi


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Using local model: /Users/nitinkumar/Work/letsTalkVoiceAgent/models/chatterbox-4bit


pkuseg not available - Chinese segmentation will be skipped


Loaded multilingual tokenizer (MTLTokenizer)
Loaded English tokenizer (EnTokenizer)
Loading S3Tokenizer from mlx-community/S3TokenizerV2...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded S3Tokenizer weights
Loaded pre-computed conditionals from conds.safetensors
Initializing Local Translation LLM at: http://127.0.0.1:1234/v1 with model: qwen3.5-0.8b


## 4. Assemble and Run Graph

In [4]:
# ---------------------------------------------------------------------------
# Graph wiring: STT -> (MT + voice_clone in parallel) -> TTS -> cleanup
# ---------------------------------------------------------------------------
workflow = StateGraph(AgentState)

workflow.add_node("stt", stt_node)
workflow.add_node("mt", mt_node)
workflow.add_node("voice_clone", voice_clone_node)
workflow.add_node("tts", tts_node)
workflow.add_node("cleanup", cleanup_node)

workflow.set_entry_point("stt")

workflow.add_edge("stt", "mt")
workflow.add_edge("stt", "voice_clone")

workflow.add_edge("mt", "tts")
workflow.add_edge("voice_clone", "tts")

workflow.add_edge("tts", "cleanup")
workflow.add_edge("cleanup", END)

app = workflow.compile()

In [5]:
async def run_agent():
    audio_input = "../inputs/recorded_audio.wav"
    output_audio_dir = "../outputs"  # treated as directory by pipeline.text_to_speech

    initial_state: AgentState = {
        "audio_path": audio_input,
        "src_lang": Language.HINDI,   # language spoken in the input audio
        "tgt_lang": Language.ENGLISH,     # desired target language
        "out_audio_path": output_audio_dir,
        "original_text": "",
        "translated_text": "",
        "ref_audio_path": None,
        "metrics": {},
    }

    print("Invoking Voice Translation Agent...")
    final_state = await app.ainvoke(initial_state)

    print(f"\nOriginal (ASR) text: {final_state['original_text']}")
    print(f"Resulting Translation: {final_state['translated_text']}")
    print("Ref audio used for cloning:", final_state.get("ref_audio_path"))
    print("Metrics:", final_state["metrics"])


# If running as a script:
# asyncio.run(run_agent())
await run_agent()


Invoking Voice Translation Agent...
--- STT NODE (MLX NATIVE) ---
Transcribing with Transformers Pipeline (HI): ../inputs/recorded_audio.wav


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


--- MT NODE (ENGLISH -> TARGET, LOCAL LLM) ---
Translating via Local LLM: Language.HINDI -> Language.ENGLISH
--- VOICE CLONE NODE (PREPARE REF CLIP) ---
Source audio duration: 12.96s
Duration > 3s; creating 7-second reference clip for voice cloning.
Wrote 5s reference clip to: ../outputs/voice_ref.wav
--- TTS NODE (MLX NATIVE, OPTIONAL VOICE CLONE) ---
Generating TTS for language: Language.ENGLISH (voice clone: ON)
Text: My name is Rahul Sharma and I am currently studying in Class 10. My Hindi exam is my top subject, and I have passed with good marks.
Voice: af_heart
Speed: 1.0x
Language: en
✅ Audio successfully generated and saving as: ../outputs/audio_000.wav
Duration:              00:00:10.279
Samples/sec:           66863.0
Prompt:                96 tokens, 26.0 tokens-per-sec
Audio:                 246720 samples, 66863.0 samples-per-sec
Real-time factor:      0.36x
Processing time:       3.69s
Peak memory usage:     1.53GB
--- CLEANUP NODE ---
Deleted temporary voice reference cli